I used this model to test out building and training the model. I used colab's GPUs.

In [ ]:
# %pip install transformers h5py pandas pyarrow protobuf

In [ ]:
# %pip install torch

# install cuda on windows
# %pip install torch --index-url https://download.pytorch.org/whl/cu126

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from pathlib import Path
from typing import List, Optional

import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


# sources where the first few turns can be text-only (hdf5_idx == -1)
_TEXT_ONLY_SOURCES = {"styletalk"}
 
# styletalk always has exactly this many text-only turns at the front of a chain
_STYLETALK_TEXT_ONLY_TURNS = 3


class AudioHDF5Dataset(Dataset):
    """
    Streams waveforms from HDF5 alongside metadata. Each item is a
    conversation chain of `num_turns` utterances in chronological order,
    resolved by walking the prev_filename linked list backwards from the
    anchor utterance (turn_index == num_turns-1 ) (zero indexed).

    Rows missing audio (hdf5_idx == -1) are excluded entirely, along with
    any chain that can't be fully resolved due to broken links.
    """

    def __init__(
        self,
        h5_path:      str,
        meta_path:    str,
        meta_columns: Optional[List[str]] = None,
        max_len_sec:  Optional[float]     = None,
        sample_rate:  int                 = 16_000,
        num_turns:    int                 = 5,
        transform=None,
    ):
        self.h5_path    = Path(h5_path)
        self.meta_path  = Path(meta_path)
        self.transform  = transform
        self.sr         = sample_rate
        self.max_len    = int(max_len_sec * sample_rate) if max_len_sec else None
        self.num_turns  = num_turns

        meta = pd.read_parquet(meta_path)
        
        # drop hard errors only; text-only rows (-1) are handled per-source below
        meta = meta[meta["hdf5_idx"] >= -1].reset_index(drop=True)

        # index by relative_audio_path so prev_filename lookups are O(1)
        self._path_to_row = meta.set_index("relative_audio_path")

        # anchor rows are the "last" turn in each chain we want to load
        anchors = meta[meta["turn_index"] == (num_turns - 1)].reset_index(drop=True)

        # resolve every anchor into a full chain, drop any with broken links
        if meta_columns is not None:
            # always keep the fields we need for chain walking + audio loading
            required = {"hdf5_idx", "relative_audio_path", "prev_filename", "turn_index"}
            self._extra_cols = [c for c in meta_columns if c not in required]
        else:
            required = {"hdf5_idx", "relative_audio_path", "prev_filename", "turn_index"}
            self._extra_cols = [c for c in meta.columns if c not in required]

        self._chains = self._build_chains(anchors)
        self._h5 = None  # lazy-open per worker so forking doesn't break the fd

    def _build_chains(self, anchors: pd.DataFrame) -> List[List[dict]]:
        chains = []
        for _, anchor_row in anchors.iterrows():
            chain = self._walk_chain(anchor_row)
            if chain is not None:
                chains.append(chain)
        return chains

    def _walk_chain(self, anchor_row) -> Optional[List[dict]]:
        # walk backwards num_turns times, collecting rows oldest-first
        chain = []
        current = anchor_row

        for _ in range(self.num_turns):
            chain.append(current)
            prev_path = current.get("prev_filename")

            # stop early if we're at the start of a conversation
            if pd.isna(prev_path) or prev_path == "":
                break

            if prev_path not in self._path_to_row.index:
                # broken link -- skip this chain entirely
                return None

            current = self._path_to_row.loc[prev_path]

        # reverse so the chain runs chronologically oldest -> newest
        chain.reverse()

        # only keep full chains -- shorter ones are incomplete conversations
        if len(chain) < self.num_turns:
            return None
 
        source = str(anchor_row.get("source", "")).lower()
 
        if source == "styletalk":
            return self._validate_styletalk_chain(chain)
        else:
            return self._validate_expresso_chain(chain)
        

    def _validate_styletalk_chain(self, chain: list) -> Optional[list]:
        # styletalk: first N turns must be text-only, the rest must have audio
        for turn_pos, row in enumerate(chain):
            hdf5_idx = int(row["hdf5_idx"])
            is_text_only_slot = turn_pos < _STYLETALK_TEXT_ONLY_TURNS
 
            if is_text_only_slot and hdf5_idx != -1:
                # we expect text-only here; a real audio file is unexpected but not fatal
                pass
            elif not is_text_only_slot and hdf5_idx == -1:
                # audio turns must actually have audio
                return None
 
        return chain
 
    def _validate_expresso_chain(self, chain: list) -> Optional[list]:
        # expresso: no text-only rows allowed anywhere in the chain
        for row in chain:
            if int(row["hdf5_idx"]) == -1:
                return None
        return chain

    def _get_h5(self):
        # open lazily so each DataLoader worker gets its own file handle
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r", swmr=True)
        return self._h5

    def _load_waveform(self, hdf5_idx: int):
        hf  = self._get_h5()
        key = f"audio/{hdf5_idx:06d}"
        ds  = hf[key]
        wav = ds[()]
        sr  = int(ds.attrs["sample_rate"])
        return wav, sr

    def _pad_or_trim(self, wav: np.ndarray) -> np.ndarray:
        if self.max_len is None:
            return wav
        if len(wav) >= self.max_len:
            return wav[: self.max_len]
        return np.pad(wav, (0, self.max_len - len(wav)))

    def __len__(self):
        return len(self._chains)

    def __getitem__(self, i):
        chain = self._chains[i]
 
        utterances = []
        for row in chain:
            hdf5_idx  = int(row["hdf5_idx"])
            is_text_only = hdf5_idx == -1
 
            if is_text_only:
                # no waveform to load; downstream should check this flag
                wav_tensor  = None
                sample_rate = self.sr
            else:
                wav, sample_rate = self._load_waveform(hdf5_idx)
 
                if self.transform is not None:
                    wav = self.transform(wav)
 
                wav        = self._pad_or_trim(wav)
                wav_tensor = torch.from_numpy(wav)
 
            utt = {
                "audio":               wav_tensor,
                "sample_rate":         sample_rate,
                "hdf5_idx":            hdf5_idx,
                "turn_index":          int(row["turn_index"]),
                "text_only":           is_text_only,
                "relative_audio_path": row.name,  # relative_audio_path is the index
            }
 
            for col in self._extra_cols:
                val = row.get(col, "")
                if isinstance(val, float) and np.isnan(val):
                    val = ""
                utt[col] = val
 
            utterances.append(utt)
 
        return utterances  # list of dicts, chronological order

    def __del__(self):
        if self._h5 is not None:
            try:
                self._h5.close()
            except Exception:
                pass


def collate_pad(batch):
    """
    Collates a batch of conversation chains. Each item in `batch` is a list
    of utterance dicts (one per turn). Returns a dict where 'audio' is
    (B, T, max_wav_len), 'lengths' is (B, T), and 'text_only' is (B, T) bool.
    Text-only turns (audio=None) are left as zero-padded rows in the tensor.
 
    Use as: DataLoader(..., collate_fn=collate_pad)
    """
    B = len(batch)
    T = len(batch[0])
 
    # only consider turns that actually have audio when computing max length
    audio_lengths = [
        utt["audio"].shape[0]
        for chain in batch
        for utt in chain
        if utt["audio"] is not None
    ]
    max_len = max(audio_lengths) if audio_lengths else 0
 
    padded    = torch.zeros(B, T, max_len, dtype=torch.float32)
    lengths   = torch.zeros(B, T, dtype=torch.long)
    text_only = torch.zeros(B, T, dtype=torch.bool)
 
    for b, chain in enumerate(batch):
        for t, utt in enumerate(chain):
            if utt["audio"] is not None:
                L = utt["audio"].shape[0]
                padded[b, t, :L] = utt["audio"]
                lengths[b, t]    = L
            text_only[b, t] = utt["text_only"]
 
    meta_keys = [k for k in batch[0][0] if k not in ("audio", "text_only")]
    out = {k: [[utt[k] for utt in chain] for chain in batch] for k in meta_keys}
 
    out["audio"]     = padded     # (B, T, max_wav_len)
    out["lengths"]   = lengths    # (B, T)
    out["text_only"] = text_only  # (B, T)
    return out

In [10]:
# from google.colab import drive
# drive.mount('/content/drive')


from torch.utils.data import DataLoader
# from ConvStyleDataset import AudioHDF5Dataset, collate_pad

# H5_PATH     = "/content/drive/MyDrive/capstone/data/merged_audio_500.h5" # <--- UPDATE THIS PATH
# META_PATH   = "/content/drive/MyDrive/capstone/data/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
H5_PATH     = "../data_TEMP/merged_audio_500.h5" # <--- UPDATE THIS PATH
META_PATH   = "../data_TEMP/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
BATCH_SIZE = 1      # Reduced batch size for memory optimization. Try 1 first. <--- MODIFIED
SAMPLE_RATE = 16_000

dataset = AudioHDF5Dataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    num_turns=5 # Changed from 5 to 1 to allow loading single-turn utterances for diagnosis
    # max_len_sec=10.0, # Uncomment and adjust this value (e.g., 10 seconds) for further memory optimization.
    # no max_len_sec -- collate_pad handles variable lengths
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    num_workers=0,
)

518.08s - invalid decimal literal (<string>, line 1)
Traceback (most recent call last):
  File "c:\Users\jackm\miniconda3\envs\capstone-model\Lib\site-packages\debugpy\_vendored\pydevd\_pydevd_bundle\pydevd_vars.py", line 629, in change_attr_expression
    value = eval(expression, frame.f_globals, frame.f_locals)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1
    [source                                                          styletalkrelative_au...               251Name: 50, dtype: object, source                                                          styletalktext_descri...250Name: music_573/c_2.wav, dtype: object]
                                                                                                             ^
SyntaxError: invalid decimal literal


In [7]:
# data batch test
batch = next(iter(loader))
print(batch.keys())
audio   = batch["audio"]           # (B, T, max_wav_len)
lengths = batch["lengths"]         # (B, T)
texts   = batch["transcription"]   # list[B][T]
wav_files = batch["relative_audio_path"]

B, T, max_wav_len = audio.shape

print(f"audio shape   : {tuple(audio.shape)}  (batch, turns, samples)")
print(f"audio dtype   : {audio.dtype}")
print(f"lengths shape : {tuple(lengths.shape)}")
print()

# print each chain as a conversation so ordering is easy to eyeball
for b in range(B):
    print(f"Chain {b}:")
    for t in range(T):
        L     = lengths[b, t].item()
        dur_s = L / dataset.sr
        text  = texts[b][t]
        file = wav_files[b][t]
        display = text[:77] + "..." if len(text) > 80 else text
        print(f"  turn {t+1}  |  {L} samples ({dur_s:.2f}s)  |  {display!r}  | {file}")
    print()

# verify no padding bleeds into real audio for any turn
for b in range(B):
    for t in range(T):
        L    = lengths[b, t].item()
        tail = audio[b, t, L:]
        if tail.numel() > 0:
            assert tail.abs().max() == 0, f"chain {b} turn {t}: non-zero past length {L}"

print("Padding check passed")

dict_keys(['sample_rate', 'hdf5_idx', 'turn_index', 'relative_audio_path', 'transcription', 'audio', 'lengths'])
audio shape   : (1, 5, 168320)  (batch, turns, samples)
audio dtype   : torch.float32
lengths shape : (1, 5)

Chain 0:
  turn 1  |  168320 samples (10.52s)  |  " okay well that looks like there's no unless they have a spam oh it smells so..."  | audio_48khz/conversational_vad_segmented/ex03-ex01/sarcastic/ex03-ex01_sarcastic_005_channel2_segment_292.5_303.02.wav
  turn 2  |  38400 samples (2.40s)  |  ' Oh, that did not just happen.'  | audio_48khz/conversational_vad_segmented/ex03-ex01/sarcastic/ex03-ex01_sarcastic_005_channel1_segment_302.71_305.11.wav
  turn 3  |  113760 samples (7.11s)  |  ' No way. Did you hear it kind of like reverberate an echo around the room as ...'  | audio_48khz/conversational_vad_segmented/ex03-ex01/sarcastic/ex03-ex01_sarcastic_005_channel2_segment_304.77_311.88.wav
  turn 4  |  115840 samples (7.24s)  |  ' I guess, you know, a voice coach must h

In [8]:
# grab one chain directly from the dataset (no collation needed for inspection)
chain_idx = 0
chain = dataset[chain_idx]

print(f"Total chains in dataset: {len(dataset)}  (num_turns={dataset.num_turns})")
print(f"\nChain {chain_idx}  ({len(chain)} turns)")
print("-" * 60)

for utt in chain:
    dur_s = utt["audio"].shape[0] / utt["sample_rate"]
    print(f"  turn {utt['turn_index']}  |  hdf5_idx={utt['hdf5_idx']}  |  {utt['audio'].shape[0]} samples ({dur_s:.2f}s)")
    for k, v in utt.items():
        if k in {"audio", "sample_rate", "hdf5_idx", "turn_index"}:
            continue
        display = str(v)
        if len(display) > 80:
            display = display[:77] + "..."
        print(f"    {k}: {display}")

print("-" * 60)

Total chains in dataset: 50  (num_turns=5)

Chain 0  (5 turns)
------------------------------------------------------------
  turn 0  |  hdf5_idx=0  |  63360 samples (3.96s)
    relative_audio_path: audio_48khz/conversational_vad_segmented/ex01-ex02/fast/ex01-ex02_fast_003_ch...
    transcription:  I like sticking little toothpicks in them and making them like soldiers.
  turn 1  |  hdf5_idx=1  |  34240 samples (2.14s)
    relative_audio_path: audio_48khz/conversational_vad_segmented/ex01-ex02/fast/ex01-ex02_fast_003_ch...
    transcription:  Yeah, and we walk them around on the table.
  turn 2  |  hdf5_idx=2  |  137920 samples (8.62s)
    relative_audio_path: audio_48khz/conversational_vad_segmented/ex01-ex02/fast/ex01-ex02_fast_003_ch...
    transcription:  Yeah, it's very cute. You roll them around. They, you know, I don't define. ...
  turn 3  |  hdf5_idx=3  |  91200 samples (5.70s)
    relative_audio_path: audio_48khz/conversational_vad_segmented/ex01-ex02/fast/ex01-ex02_fast_003_

In [ ]:
from transformers import BertTokenizer, BertModel, WavLMModel, Wav2Vec2FeatureExtractor

bert_tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model      = BertModel.from_pretrained("bert-base-uncased").half().to(device)  # using half() to limit gpu memory for quick tests

# WavLM only needs a feature extractor, not a full processor with tokenizer
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model     = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").half().to(device)

for model in (bert_model, wavlm_model):
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("Encoders loaded and frozen")

In [ ]:
class SelfAttentivePooling(nn.Module):

    def __init__(self, dim: int):
        super().__init__()
        self.scorer = nn.Linear(dim, 1, bias=False)

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # hidden: (B, T, d),  mask: (B, T) -- 1 for real tokens, 0 for padding
        scores = self.scorer(hidden).squeeze(-1)  # (B, T)

        if mask is not None:
            # push padding positions to -inf so softmax zeroes them out
            scores = scores.masked_fill(~mask.bool(), float("-inf"))

        weights = F.softmax(scores, dim=-1)
        return (weights.unsqueeze(-1) * hidden).sum(dim=1)  # (B, d)

In [ ]:
class ModalityEncoder(nn.Module):

    def __init__(self, backbone, pooler):
        super().__init__()
        self.backbone = backbone
        self.pooler   = pooler

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        return self.pooler(hidden, mask)  # (B, d)

In [ ]:
def encode_text_inputs(texts, tokenizer):
    return tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

def encode_audio_inputs(audio, lengths, processor):
    inputs = processor(
        list(audio.cpu().numpy()),
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
    ).to(device)
    return inputs, lengths

In [ ]:
class DualModalityEmbedder(nn.Module):

    def __init__(self, text_encoder, audio_encoder, tokenizer, processor):
        super().__init__()
        self.text_encoder  = text_encoder
        self.audio_encoder = audio_encoder
        self.tokenizer     = tokenizer
        self.processor     = processor

    def embed_text(self, texts):
        tokens = encode_text_inputs(texts, self.tokenizer)
        with torch.no_grad():
            hidden = self.text_encoder.backbone(**tokens).last_hidden_state
        return self.text_encoder(hidden, tokens["attention_mask"])

    def embed_audio(self, audio, lengths):
        inputs, lengths = encode_audio_inputs(audio, lengths, self.processor)
        with torch.no_grad():
            hidden = self.audio_encoder.backbone(**inputs).last_hidden_state

        # resize the mask to match WavLM's downsampled time dimension
        T_out    = hidden.shape[1]
        raw_mask = (torch.arange(audio.shape[1], device=device).unsqueeze(0)
                    < lengths.unsqueeze(1).to(device)).float()
        mask_ds  = F.interpolate(raw_mask.unsqueeze(1), size=T_out, mode="nearest").squeeze(1).bool()

        return self.audio_encoder(hidden, mask_ds)

    def forward(self, audio, lengths, texts):
        # convenience method, downstream transformers should call embed_text/embed_audio directly
        return self.embed_text(texts), self.embed_audio(audio, lengths)


# CHANGED TO HALF
text_encoder  = ModalityEncoder(bert_model,  SelfAttentivePooling(768).half()).to(device)
audio_encoder = ModalityEncoder(wavlm_model, SelfAttentivePooling(768).half()).to(device)

embedder = DualModalityEmbedder(
    text_encoder, audio_encoder, bert_tokenizer, wavlm_processor
).to(device)

print("Embedder ready")

In [ ]:
embedder.eval()

# Re-define batch here as it was not found in the kernel state
batch = next(iter(loader))

wavlm_model.to(device)
audio_emb = embedder.embed_audio(batch["audio"].to(device).half(), batch["lengths"])
wavlm_model.to("cpu")
torch.cuda.empty_cache()

bert_model.to(device)
text_emb = embedder.embed_text(batch["transcription"])
bert_model.to("cpu")
torch.cuda.empty_cache()

print("text_emb shape :", text_emb.shape)
print("audio_emb shape:", audio_emb.shape)

assert text_emb.shape  == (BATCH_SIZE, 768)
assert audio_emb.shape == (BATCH_SIZE, 768)
assert not torch.allclose(text_emb, audio_emb), "embeddings are suspiciously identical"

print("All assertions passed")